# 3 — Best Model Refinement
## Select the Best Candidate and Improve It (Short Run)

This notebook refines only one model to keep runtime reasonable.

Workflow:
1. Load data and models
2. Select best model from UltraFast tuning summary (if available)
3. Retrain the selected model with more epochs
4. Visualize refinement curves and quick validation outputs

In [ ]:
import json, numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

import torch
from torch.utils.data import DataLoader

import sys, os
sys.path.insert(0, os.path.abspath('..'))

from config import MODEL_CONFIG, TRAIN_CONFIG, SEED
from src.data.dataset import FireSpreadDataset
from src.models.cellular_automata import CellularAutomataModel
from src.models.convlstm import ConvLSTMModel
from src.models.unet import UNetFire
from src.models.pi_cca import PIConvCellularAutomaton
from src.training.trainer import Trainer

# ── Charger la config générée par 00_Setup.ipynb ──────────────────────────────
_cfg_path = Path().resolve() / "setup_config.json"
if not _cfg_path.exists():
    raise FileNotFoundError(
        "setup_config.json introuvable.\n"
        "→ Lance d'abord le notebook 00_Setup.ipynb"
    )
cfg = json.load(open(_cfg_path))

PROCESSED_DIR    = Path(cfg["PROCESSED_DIR"])
FIGURES_DIR      = Path(cfg["FIGURES_DIR"])
MODELS_DIR       = Path(cfg["MODELS_DIR"])
RESULTS_DIR      = Path(cfg["RESULTS_DIR"]) if "RESULTS_DIR" in cfg else Path("../results")
FEATURE_CHANNELS = cfg["FEATURE_CHANNELS"]
N_INPUT_CHANNELS = cfg["N_INPUT_CHANNELS"]
CH               = cfg["CH"]
GRID_SIZE        = cfg["GRID_SIZE"]
norm_stats       = cfg["norm_stats"]

def ensure_npz_key_format(processed_dir):
    processed_dir = Path(processed_dir)
    for split in ["train", "val", "test"]:
        p = processed_dir / f"{split}.npz"
        if not p.exists():
            continue
        d = np.load(p)
        keys = set(d.files)
        if {"inputs", "targets"}.issubset(keys):
            continue
        if {"X", "Y"}.issubset(keys):
            x = d["X"]
            y = d["Y"]
            np.savez(p, inputs=x, targets=y)
            print(f"Converted {p.name}: X/Y -> inputs/targets")
        else:
            print(f"Warning: unexpected keys in {p.name}: {sorted(keys)}")

TRAINING_CONFIG = TRAIN_CONFIG.copy()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE = str(device)
print(f"Config chargée — Device : {DEVICE}")

## 3.1 Load Data

In [ ]:
# Fallback si la cellule d'imports n'a pas été relancée après restart kernel
if 'ensure_npz_key_format' not in globals():
    from pathlib import Path
    import numpy as np
    def ensure_npz_key_format(processed_dir):
        processed_dir = Path(processed_dir)
        for split in ["train", "val", "test"]:
            p = processed_dir / f"{split}.npz"
            if not p.exists():
                continue
            d = np.load(p)
            keys = set(d.files)
            if {"inputs", "targets"}.issubset(keys):
                continue
            if {"X", "Y"}.issubset(keys):
                np.savez(p, inputs=d["X"], targets=d["Y"])
                print(f"Converted {p.name}: X/Y -> inputs/targets")
            else:
                print(f"Warning: unexpected keys in {p.name}: {sorted(keys)}")

ensure_npz_key_format(PROCESSED_DIR)

# Notebook-only loading: robust stats + CPU/GPU-aware pin_memory
train_npz = np.load(PROCESSED_DIR / 'train.npz')
if 'inputs' in train_npz.files:
    train_inputs = train_npz['inputs']
else:
    train_inputs = train_npz['X']

# Per-channel stats from train split to avoid default-stats mismatch
channel_means = train_inputs.mean(axis=(0, 2, 3))
channel_stds = train_inputs.std(axis=(0, 2, 3)) + 1e-8
computed_stats = {
    FEATURE_CHANNELS[i]: {'mean': float(channel_means[i]), 'std': float(channel_stds[i])}
    for i in range(len(FEATURE_CHANNELS))
}

datasets = {
    'train': FireSpreadDataset(PROCESSED_DIR, split='train', augment=True,  stats=computed_stats, seed=SEED),
    'val':   FireSpreadDataset(PROCESSED_DIR, split='val',   augment=False, stats=computed_stats, seed=SEED),
    'test':  FireSpreadDataset(PROCESSED_DIR, split='test',  augment=False, stats=computed_stats, seed=SEED),
}

pin_mem = torch.cuda.is_available()
loaders = {
    split: DataLoader(
        ds,
        batch_size=TRAINING_CONFIG['batch_size'],
        shuffle=(split == 'train'),
        num_workers=0,
        pin_memory=pin_mem,
        drop_last=(split == 'train'),
    )
    for split, ds in datasets.items()
}

for name, loader in loaders.items():
    print(f'{name}: {len(loader.dataset)} samples, {len(loader)} batches')

# Verify shapes
x_batch, y_batch = next(iter(loaders['train']))
print(f'\nBatch shapes: X={x_batch.shape}, Y={y_batch.shape}')
print(f'X range: [{x_batch.min():.2f}, {x_batch.max():.2f}]')
print(f'Y unique: {torch.unique(y_batch).tolist()}')

## 3.2 Instantiate Models

Each model is defined in `src/models/` and configured via `config.MODEL_CONFIG`.

In [ ]:
MODEL_CLASSES = {
    'ca': CellularAutomataModel,
    'convlstm': ConvLSTMModel,
    'unet': UNetFire,
    'pi_cca': PIConvCellularAutomaton,
}

for name, cls in MODEL_CLASSES.items():
    model = cls(config=MODEL_CONFIG[name])
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'{name:>10s}: {MODEL_CONFIG[name]["name"]:>35s}  |  {n_params:>10,d} params')

## 3.3 ConvLSTM Direct Training (No CV / No Tuning)

This section trains **ConvLSTM only** with fixed, stable hyperparameters for fast comparison with the other models.

### Protocol

- No cross-validation
- No hyperparameter search
- Single direct training on train set, validation on val set

### Fixed hyperparameters

- `learning_rate = 1e-4`
- `weight_decay = 1e-5`
- `focal_alpha = 0.25`
- `focal_gamma = 2.0`
- `dropout = 0.10`

In [ ]:
def build_model(model_name, overrides=None):
    cfg = dict(MODEL_CONFIG[model_name])
    if overrides:
        cfg.update(overrides)
    return MODEL_CLASSES[model_name](config=cfg).to(device), cfg

is_cpu = (str(device) == 'cpu')
model_name = 'convlstm'

# Fixed ConvLSTM hyperparameters (no CV / no tuning)
fixed_hp = {
    'learning_rate': 1e-4,
    'weight_decay': 1e-5,
    'focal_alpha': 0.25,
    'focal_gamma': 2.0,
    'dropout': 0.10,
}

# Keep runtime reasonable
train_epochs = 8 if is_cpu else 20
train_patience = 2 if is_cpu else 5

print(f'Direct ConvLSTM training on {device}')
print(f'Fixed hyperparameters: {fixed_hp}')

# Runtime estimate based on your observed CPU logs (~14.5 min / epoch)
epoch_minutes = 14.5 if is_cpu else 0.25
est_total_min = train_epochs * epoch_minutes
est_low_h = (est_total_min * 0.75) / 60.0
est_high_h = (est_total_min * 1.25) / 60.0
print(f'Estimated runtime (ConvLSTM direct): ~{est_low_h:.1f}h to ~{est_high_h:.1f}h')

model, _ = build_model(model_name, {'dropout': fixed_hp['dropout']})
cfg_train = TRAINING_CONFIG.copy()
cfg_train.update({
    'epochs': train_epochs,
    'learning_rate': fixed_hp['learning_rate'],
    'weight_decay': fixed_hp['weight_decay'],
    'focal_alpha': fixed_hp['focal_alpha'],
    'focal_gamma': fixed_hp['focal_gamma'],
    'early_stopping_patience': train_patience,
})

trainer = Trainer(model=model, model_name=model_name, device=device, config=cfg_train)
direct_history = trainer.train(loaders['train'], loaders['val'])

save_dir = MODELS_DIR / model_name
save_dir.mkdir(parents=True, exist_ok=True)
with open(save_dir / 'convlstm_direct_history.json', 'w') as f:
    json.dump(direct_history, f, indent=2, default=str)

direct_report = {
    'model': model_name,
    'mode': 'direct_no_cv_no_tuning',
    'hyperparameters': fixed_hp,
    'epochs': train_epochs,
    'early_stopping_patience': train_patience,
    'estimated_runtime_hours': {'low': est_low_h, 'high': est_high_h},
}
with open(RESULTS_DIR / 'convlstm_direct_report.json', 'w') as f:
    json.dump(direct_report, f, indent=2)

print('\nDirect ConvLSTM training complete.')

## 3.4 ConvLSTM Training Curves

In [ ]:
# Load direct ConvLSTM history
hist_path = MODELS_DIR / 'convlstm' / 'convlstm_direct_history.json'
if not hist_path.exists():
    print('No ConvLSTM direct history found. Run Cell 8 first.')
else:
    with open(hist_path) as f:
        hist = json.load(f)

    label = MODEL_CONFIG['convlstm']['name']
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(hist['train_loss'], label=f'{label} (train)', color='#1f77b4', linestyle='-')
    axes[0].plot(hist['val_loss'], label=f'{label} (val)', color='#ff7f0e', linestyle='--')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss (Focal + Dice)')
    axes[0].set_title('ConvLSTM Direct Training Loss', fontweight='bold')
    axes[0].legend(fontsize=9)

    if 'val_metrics' in hist and len(hist['val_metrics']) > 0:
        val_dice = [m.get('dice', np.nan) for m in hist['val_metrics']]
        axes[1].plot(val_dice, label=f'{label} (Dice)', color='#2ca02c')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Validation Dice')
    axes[1].set_title('ConvLSTM Validation Dice', fontweight='bold')
    axes[1].legend(fontsize=9)

    plt.tight_layout()
    plt.savefig('../results/figures/convlstm_direct_curves.png', dpi=150, bbox_inches='tight')
    plt.show()

## 3.5 Quick Validation Check (ConvLSTM)

In [ ]:
# Visualise ConvLSTM prediction on a validation batch
x_val, y_val = next(iter(loaders['val']))
x_val, y_val = x_val.to(device), y_val.to(device)

model = MODEL_CLASSES['convlstm'](config=MODEL_CONFIG['convlstm']).to(device)
ckpt = MODELS_DIR / 'convlstm' / 'best_model.pt'
if ckpt.exists():
    model.load_state_dict(torch.load(ckpt, map_location=device))
model.eval()

with torch.no_grad():
    pred = model(x_val[:1]).squeeze().cpu().numpy()

gt = y_val[0].squeeze().cpu().numpy()
fire_in = x_val[0, -1].cpu().numpy()

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(fire_in, cmap='hot', vmin=0, vmax=1)
axes[0].set_title('Input Fire')

axes[1].imshow(pred, cmap='hot', vmin=0, vmax=1)
axes[1].set_title('Prediction (ConvLSTM)')

axes[2].imshow(gt, cmap='hot', vmin=0, vmax=1)
axes[2].set_title('Ground Truth')

diff = np.abs(pred - gt)
axes[3].imshow(diff, cmap='Reds', vmin=0, vmax=1)
axes[3].set_title('|Error|')

for ax in axes:
    ax.axis('off')

plt.suptitle('ConvLSTM Direct — Validation Sample', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/figures/convlstm_direct_val_prediction.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary

- This notebook trains ConvLSTM directly with fixed hyperparameters.
- No CV and no hyperparameter tuning are used.
- Outputs are saved in `convlstm_direct_report.json` and `convlstm_direct_history.json`.

Use Notebook 04 for final test-set comparison with the other models.